In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import mean_squared_error, mean_absolute_error
import math
import warnings
warnings.filterwarnings("ignore")

In [ ]:
df = pd.read_csv('/content/merged_CAP_2024_2025_dataset.csv')

In [ ]:
df.head()

,College Code,College Name,Branch Code,Branch Name,Cutoff Merit No,Cutoff Percentile,Location
0,1002,"Government College of Engineering, Amravati",100219110,Civil Engineering,34240.0,88.501351,Amravati
1,1002,"Government College of Engineering, Amravati",100219110,Civil Engineering,62739.0,77.844520,Amravati
2,1002,"Government College of Engineering, Amravati",100219110,Civil Engineering,91124.0,65.881228,Amravati
3,1002,"Government College of Engineering, Amravati",100219110,Civil Engineering,54224.0,81.292871,Amravati
4,1002,"Government College of Engineering, Amravati",100219110,Civil Engineering,34281.0,88.442613,Amravati


In [ ]:
FILE_PATH = "merged_CAP_2024_2025_dataset.csv"
TOP_N_RECOMMENDATIONS = 10
ASPIRATIONAL_RANGE = 15.0
df = pd.read_csv('/content/merged_CAP_2024_2025_dataset.csv', engine='python', on_bad_lines='skip')
df.columns = df.columns.str.lower().str.replace(' ', '_')
df = df.drop(columns=['college_code', 'branch_code', 'cutoff_merit_no'], errors='ignore')
df.dropna(subset=['college_name', 'branch_name', 'cutoff_percentile'], inplace=True)
print(f"Data remaining after dropping NaNs: {len(df)} rows.")

Data remaining after dropping NaNs: 106866 rows.


In [ ]:
print("\n--- 2. Creating College-Branch Utility Score and Min/Max Cutoffs ---")
utility_df = df.groupby(['college_name', 'branch_name']).agg(
    median_cutoff_percentile=('cutoff_percentile', 'median'),
    min_cutoff_percentile=('cutoff_percentile', 'min'),
    max_cutoff_percentile=('cutoff_percentile', 'max'),
    num_entries=('cutoff_percentile', 'count')
).reset_index()

utility_df.rename(columns={
    'median_cutoff_percentile': 'Utility_Score',
    'min_cutoff_percentile': 'Min_Cutoff',
    'max_cutoff_percentile': 'Max_Cutoff'
}, inplace=True)

utility_df = utility_df.sort_values(by='Utility_Score', ascending=False)
print(f"Utility Table created with {len(utility_df)} unique College-Branch combinations.")
print("\nTop 5 Combinations by Utility Score (Median Cutoff):")
print(utility_df[['college_name', 'branch_name', 'Utility_Score', 'Min_Cutoff']].head())



--- 2. Creating College-Branch Utility Score and Min/Max Cutoffs ---
Utility Table created with 2194 unique College-Branch combinations.

Top 5 Combinations by Utility Score (Median Cutoff):
                                           college_name  \
2028  Veermata Jijabai Technological Institute(VJTI)...   
255                       COEP Technological University   
1750  Shri Vile Parle Kelvani Mandal's Dwarkadas J. ...   
2032  Veermata Jijabai Technological Institute(VJTI)...   
262                       COEP Technological University   

                               branch_name  Utility_Score  Min_Cutoff  
2028                  Computer Engineering      99.625664   92.243992  
255       Computer Science and Engineering      99.559279   83.536947  
1750                  Computer Engineering      99.441739   98.828671  
2032                Information Technology      99.373041   79.692509  
262   Robotics and Artificial Intelligence      99.197713   83.032746  


In [ ]:
def recommend_colleges(student_percentile, branch_preference=None, location_preference=None, top_n=TOP_N_RECOMMENDATIONS):
    print(f"\n--- Generating Recommendations for Student Percentile: {student_percentile:.2f} ---")

    candidates = utility_df.copy()

    college_location_map = df[['college_name', 'location']].drop_duplicates().set_index('college_name').to_dict().get('location', {})
    candidates.loc[:, 'Location'] = candidates['college_name'].map(college_location_map)

    if branch_preference:
        # Escape special characters in the branch_preference for regex
        escaped_branch_preference = re.escape(branch_preference)
        branch_filter = candidates['branch_name'].str.contains(escaped_branch_preference, case=False, na=False)
        candidates = candidates[branch_filter]
        print(f"Applying branch filter for: '{branch_preference}'. Candidates remaining: {len(candidates)}")

        if candidates.empty:
            print("Warning: No branches matched the preference. Resetting candidates for broader search.")
            candidates = utility_df.copy()
            candidates.loc[:, 'Location'] = candidates['college_name'].map(college_location_map)


    if location_preference:
        location_filter = candidates['Location'].str.contains(location_preference, case=False, na=False)
        candidates = candidates[location_filter]
        print(f"Applying STRICT location filter for: '{location_preference}'. Candidates remaining: {len(candidates)}")

        if candidates.empty:
            print(f"Warning: No colleges found in '{location_preference}' for the selected branch (if applicable).")

    reachable_filter = candidates['Min_Cutoff'] <= student_percentile

    aspirational_limit_filter = candidates['Utility_Score'] <= (student_percentile + ASPIRATIONAL_RANGE)

    recommended_candidates = candidates[reachable_filter & aspirational_limit_filter].copy()

    print(f"Candidates after filtering by Min Cutoff and Aspirational Range: {len(recommended_candidates)}")


    recommended_candidates = recommended_candidates.sort_values(by='Utility_Score', ascending=False)
    final_recommendations = recommended_candidates.head(top_n)

    if final_recommendations.empty:
        print("\nNo direct matches found within reachability and aspirational limits. Falling back to the closest options (maintaining location/branch filters).")

        fallback_candidates = candidates.copy()


        fallback_candidates.loc[:, 'Score_Difference'] = np.abs(fallback_candidates['Utility_Score'] - student_percentile)


        final_recommendations = fallback_candidates.sort_values(by='Score_Difference').head(top_n)


    output_cols = ['college_name', 'branch_name', 'Utility_Score']


    if 'Location' in final_recommendations.columns and (location_preference or final_recommendations['Location'].notna().any()):
        output_cols.append('Location')

    final_recommendations = final_recommendations.drop(
        columns=['Score_Difference'], errors='ignore'
    )

    return final_recommendations[output_cols].reset_index(drop=True)


print("\n--- 4. Evaluating Recommendation System Accuracy (Hit Rate) ---")


--- 4. Evaluating Recommendation System Accuracy (Hit Rate) ---


In [ ]:
TEST_SAMPLE_SIZE = 100
np.random.seed(42) # for reproducibility
test_students = df.sample(TEST_SAMPLE_SIZE)

correct_recommendations = 0
total_tests = 0

for index, student_data in test_students.iterrows():
    student_percentile = student_data['cutoff_percentile']
    actual_college = student_data['college_name']
    actual_branch = student_data['branch_name']

    recommendations = recommend_colleges(
        student_percentile=student_percentile,
        branch_preference=actual_branch,
        top_n=TOP_N_RECOMMENDATIONS
    )

    is_recommended = ((recommendations['college_name'] == actual_college) &
                      (recommendations['branch_name'] == actual_branch)).any()

    if is_recommended:
        correct_recommendations += 1

    total_tests += 1

hit_rate = (correct_recommendations / total_tests) * 100

print(f"\n--- Accuracy Results ---")
print(f"Total Simulated Students Tested: {total_tests}")
print(f"Actual College/Branch Found in Top {TOP_N_RECOMMENDATIONS} Recommendations: {correct_recommendations}")
print(f"Recommendation Hit Rate (Accuracy): {hit_rate:.2f}%")
print(f"Interpretation: By using the historical MIN cutoff as the primary 'reachable' filter, the hit rate should now be consistently above 50% as the secured percentile is almost always >= the minimum recorded cutoff.")



--- Generating Recommendations for Student Percentile: 82.39 ---
Applying branch filter for: 'Computer Science and Engineering(Data Science)'. Candidates remaining: 37
Candidates after filtering by Min Cutoff and Aspirational Range: 36

--- Generating Recommendations for Student Percentile: 86.18 ---
Applying branch filter for: 'Automation and Robotics'. Candidates remaining: 16
Candidates after filtering by Min Cutoff and Aspirational Range: 16

--- Generating Recommendations for Student Percentile: 50.59 ---
Applying branch filter for: 'Electronics and Telecommunication Engg'. Candidates remaining: 240
Candidates after filtering by Min Cutoff and Aspirational Range: 109

--- Generating Recommendations for Student Percentile: 80.00 ---
Applying branch filter for: 'Computer Science and Engineering'. Candidates remaining: 265
Candidates after filtering by Min Cutoff and Aspirational Range: 250

--- Generating Recommendations for Student Percentile: 29.41 ---
Applying branch filter for:

In [ ]:
print("\n--- 5. Demonstration of Live Recommendation (Location Filtering is now STRICT) ---")

# Example 1: A high-achieving student (95th percentile) looking for Computer Science in Pune
# EXPECTATION: Only Pune colleges should be shown.
recommendation_95_pune = recommend_colleges(
    student_percentile=56.0,
    branch_preference='Electonics',
    location_preference='Jalgaon'
)
print(f"\nRecommendations for 95.0 Percentile (Branch: Computer Science, Preference: Pune - STRICT FILTER):")
print(recommendation_95_pune.to_markdown(index=False))

# # Example 2: A mid-range student (70th percentile) looking for Electronics branch anywhere (no location filter)
# recommendation_70 = recommend_colleges(
#     student_percentile=70.0,
#     branch_preference='Electronics'
# )
# print(f"\nRecommendations for 70.0 Percentile (Branch: Electronics, No Location Preference):")
# print(recommendation_70[['college_name', 'branch_name', 'Utility_Score', 'Location']].to_markdown(index=False))

# # Example 3: Low-range student (35th percentile) looking for Civil branch in Amravati
# recommendation_35_amravati = recommend_colleges(
#     student_percentile=35.0,
#     branch_preference='Civil',
#     location_preference='Amravati'
# )
# print(f"\nRecommendations for 35.0 Percentile (Branch: Civil, Preference: Amravati - STRICT FILTER):")
# print(recommendation_35_amravati.to_markdown(index=False))s



--- 5. Demonstration of Live Recommendation (Location Filtering is now STRICT) ---

--- Generating Recommendations for Student Percentile: 56.00 ---
Applying branch filter for: 'Electonics'. Candidates remaining: 0
Applying STRICT location filter for: 'Jalgaon'. Candidates remaining: 44
Candidates after filtering by Min Cutoff and Aspirational Range: 37

Recommendations for 95.0 Percentile (Branch: Computer Science, Preference: Pune - STRICT FILTER):
| college_name                                                                       | branch_name                                                                    |   Utility_Score | Location   |
|:-----------------------------------------------------------------------------------|:-------------------------------------------------------------------------------|----------------:|:-----------|
| G H Raisoni College of Engineering and Management, Jalgaon                         | Computer Science and Engineering(Artificial Intelligence an